# Liver Disease Q&A Fine-Tuning — Fixed QLoRA Notebook

This version fixes the main problems in the uploaded notebook:

- Uses **one consistent base model** (`microsoft/Phi-3-mini-4k-instruct`).
- Does not mix a local model folder with an unrelated Hugging Face model/config.
- Uses a proper Hugging Face model download for Google Colab.
- Handles newer/older TRL `SFTConfig` argument names.
- Avoids hard-coded Windows paths in Colab.
- Checks the JSONL dataset structure before training.
- Uses QLoRA only when CUDA is available.
- Saves the LoRA adapter and tokenizer.


In [1]:
# Cell 1 — Check environment
import sys, os, subprocess, torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
    try:
        print(subprocess.check_output(["nvidia-smi"], text=True))
    except Exception:
        pass
else:
    print("WARNING: No CUDA GPU detected. QLoRA training is not recommended on CPU.")


Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
PyTorch: 2.10.0+cpu
CUDA available: False


In [ ]:
# Cell 2 — Install dependencies
# Run this cell in Google Colab.
# Restart the runtime only if Colab asks you to.

!pip -q install -U transformers datasets accelerate peft trl bitsandbytes sentencepiece


In [2]:
# Cell 3 — Imports and versions
import json
import os
import sys
import inspect
import torch

import transformers
import datasets
import peft
import trl

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)


c:\Users\tagbi\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers: 4.57.6
datasets: 5.0.1
peft: 0.20.0
trl: 1.9.2


In [3]:
# ── Cell 3: Load dataset (Colab Upload vs Local Paths) ───────────────────────
import sys
import os

is_colab = 'google.colab' in sys.modules
dataset_path = None

if is_colab:
    from google.colab import files
    print("Running on Google Colab. Please upload your 'liver_qa_dataset.jsonl' file...")
    uploaded = files.upload()
    if uploaded:
        dataset_path = list(uploaded.keys())[0]
        print(f"Uploaded successfully: {dataset_path}")
else:
    print("Running locally. Searching for liver_qa_dataset.jsonl in workspace paths...")
    # Standard project directories to check relative to notebook
    candidate_paths = [
        '../Data/liver_qa_dataset.jsonl',
        'Data/liver_qa_dataset.jsonl',
        './liver_qa_dataset.jsonl',
        './Data/liver_qa_dataset.jsonl'
    ]
    for path in candidate_paths:
        if os.path.exists(path):
            dataset_path = path
            break
            
    if dataset_path:
        print(f"Found local dataset at: {os.path.abspath(dataset_path)}")
    else:
        dataset_path = input("Could not find dataset automatically. Please enter absolute or relative path to liver_qa_dataset.jsonl: ")
        if not os.path.exists(dataset_path):
            raise FileNotFoundError(f"Specified dataset path does not exist: {dataset_path}")

print(f"Active dataset path: {dataset_path}")

Running locally. Searching for liver_qa_dataset.jsonl in workspace paths...
Found local dataset at: f:\Liver Disease Detectioon system 2\Data\liver_qa_dataset.jsonl
Active dataset path: ../Data/liver_qa_dataset.jsonl


In [4]:
# Cell 5 — Validate and load JSONL
records = []

with open(dataset_path, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            item = json.loads(line)
        except json.JSONDecodeError as e:
            raise ValueError(f"Invalid JSON on line {line_no}: {e}")
        records.append(item)

if not records:
    raise ValueError("Dataset is empty.")

def validate_record(item, index):
    if "messages" not in item:
        raise ValueError(f"Example {index} has no 'messages' field.")
    if not isinstance(item["messages"], list) or not item["messages"]:
        raise ValueError(f"Example {index} has an invalid/empty 'messages' list.")

    for j, msg in enumerate(item["messages"]):
        if not isinstance(msg, dict):
            raise ValueError(f"Example {index}, message {j} is not an object.")
        if msg.get("role") not in {"system", "user", "assistant"}:
            raise ValueError(
                f"Example {index}, message {j}: invalid role {msg.get('role')!r}"
            )
        if "content" not in msg:
            raise ValueError(f"Example {index}, message {j} has no content.")

for i, item in enumerate(records):
    validate_record(item, i)

print("Examples:", len(records))
print("\nFirst example:")
for msg in records[0]["messages"]:
    print(f"[{msg['role']}] {msg['content'][:300]}")


Examples: 341

First example:
[system] You are LiverAI — a friendly, easy-to-understand liver health assistant. Your goal is to help patients, caregivers, and students learn about liver diseases in a clear and simple way. You are trained on medical references including NCBI StatPearls, AASLD guidelines, and liver disease textbooks.

Use 
[user] What leads to Autoimmune Causes?
[assistant] Autoimmune Hepatitis (AIH): - Destruction of liver parenchyma by autoantibodies - More common in females - Lab: Elevated ANA, anti-smooth muscle antibodies (ASMA), LKM-1, hypergammaglobulinemia - Treatment: Corticosteroids (prednisone), azathioprine Primary Biliary Cholangitis (PBC): - Autoimmune de


## Important model-path fix

The original notebook mixed:

- `microsoft/Phi-3-mini-4k-instruct` in the configuration,
- `unsloth/llama-3-8b-Instruct` as the tokenizer/config source,
- and a hard-coded Windows folder as the model weights.

Those three must **not** be mixed.

This notebook uses the same Hugging Face model for tokenizer, config, and weights.


In [5]:
# Cell 6 — Configuration
BASE_MODEL = "microsoft/Phi-3-mini-4k-instruct"
OUTPUT_DIR = "/content/liver-lora"

EPOCHS = 3
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUM = 8
LEARNING_RATE = 2e-4
MAX_LENGTH = 1024

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

VAL_SPLIT = 0.10
SEED = 42

print("Base model:", BASE_MODEL)
print("Output:", OUTPUT_DIR)


Base model: microsoft/Phi-3-mini-4k-instruct
Output: /content/liver-lora


In [6]:
# Cell 7 — Load tokenizer and model with QLoRA
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

use_cuda = torch.cuda.is_available()

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

model_kwargs = {
    "trust_remote_code": True,
}

if use_cuda:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model_kwargs.update({
        "quantization_config": bnb_config,
        "device_map": "auto",
        "torch_dtype": torch.float16,
    })

    print("Loading 4-bit QLoRA model...")
else:
    # CPU fallback: this can require a large amount of RAM and is very slow.
    model_kwargs["torch_dtype"] = torch.float32
    print("Loading full-precision CPU model. Training may be extremely slow.")

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    **model_kwargs
)

model.config.use_cache = False

print("Model loaded successfully.")
print("Model device:", next(model.parameters()).device)


Loading full-precision CPU model. Training may be extremely slow.


`torch_dtype` is deprecated! Use `dtype` instead!
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.
Loading checkpoint shards: 100%|██████████| 2/2 [00:43<00:00, 21.57s/it]


Model loaded successfully.
Model device: cpu


In [7]:
# Cell 8 — Apply LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

if use_cuda:
    model = prepare_model_for_kbit_training(model)

# Phi-3 uses these projection names.
target_modules = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_up_proj",
    "down_proj",
]

# Keep only modules that actually exist in this model.
existing = set()
for name, module in model.named_modules():
    short_name = name.split(".")[-1]
    existing.add(short_name)

target_modules = [m for m in target_modules if m in existing]

if not target_modules:
    raise RuntimeError(
        "Could not find the expected Phi-3 LoRA target modules. "
        "Print model.named_modules() and update target_modules."
    )

print("LoRA target modules:", target_modules)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=target_modules,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


LoRA target modules: ['o_proj', 'gate_up_proj', 'down_proj']
trainable params: 18,874,368 || all params: 3,839,953,920 || trainable%: 0.4915


In [8]:
# Cell 9 — Format dataset
from datasets import Dataset

def format_example(example):
    messages = example["messages"]

    try:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        )
    except Exception:
        # Fallback only if the tokenizer has no usable chat template.
        parts = []
        for m in messages:
            parts.append(
                f"<|{m['role']}|>\n{m['content']}"
            )
        text = "\n".join(parts)

    return {"text": text}

raw_ds = Dataset.from_list(records)
formatted_ds = raw_ds.map(
    format_example,
    remove_columns=raw_ds.column_names
)

split = formatted_ds.train_test_split(
    test_size=VAL_SPLIT,
    seed=SEED
)

print("Train:", len(split["train"]))
print("Validation:", len(split["test"]))
print("\nSample:")
print(split["train"][0]["text"][:1000])


Map: 100%|██████████| 341/341 [00:02<00:00, 149.00 examples/s]


Train: 306
Validation: 35

Sample:
<|system|>
You are LiverAI — a friendly, easy-to-understand liver health assistant. Your goal is to help patients, caregivers, and students learn about liver diseases in a clear and simple way. You are trained on medical references including NCBI StatPearls, AASLD guidelines, and liver disease textbooks.

Use simple, everyday language. Use bullet points for lists of symptoms, causes, or treatments. Always end with: 'Please talk to your doctor for advice specific to you.'<|end|>
<|user|>
copper liver<|end|>
<|assistant|>
Two important genetic liver diseases: 

**Wilson Disease:** Autosomal recessive disorder causing copper accumulation (ATP7B gene mutation). Features: Liver disease, neuropsychiatric symptoms, Kayser-Fleischer rings (copper deposits in cornea). Diagnosis: Low serum ceruloplasmin, elevated 24-hour urine copper. Treatment: Copper chelators (penicillamine, trientine), zinc acetate; liver transplant in liver failure. 

**Hereditary Hemochro

In [9]:
# Cell 10 — Check token lengths before training
lengths = []

for item in split["train"]:
    tokenized = tokenizer(
        item["text"],
        truncation=False,
        add_special_tokens=True
    )
    lengths.append(len(tokenized["input_ids"]))

print("Maximum tokens:", max(lengths))
print("Average tokens:", sum(lengths) / len(lengths))
print("Examples longer than MAX_LENGTH:", sum(x > MAX_LENGTH for x in lengths))

if max(lengths) > MAX_LENGTH:
    print(
        f"WARNING: Some examples exceed {MAX_LENGTH} tokens and will be truncated."
    )


Maximum tokens: 811
Average tokens: 312.37908496732024
Examples longer than MAX_LENGTH: 0


In [10]:
# Cell 11 — Build TRL SFTConfig with compatibility handling
from trl import SFTTrainer, SFTConfig

os.makedirs(OUTPUT_DIR, exist_ok=True)

optim_name = "paged_adamw_8bit" if use_cuda else "adamw_torch"

available = inspect.signature(SFTConfig.__init__).parameters

kwargs = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": EPOCHS,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM,
    "learning_rate": LEARNING_RATE,
    "weight_decay": 0.01,
    "warmup_ratio": 0.05,
    "lr_scheduler_type": "cosine",
    "optim": optim_name,
    "logging_steps": 10,
    "save_strategy": "epoch",
    "save_total_limit": 1,
    "load_best_model_at_end": True,
    "fp16": bool(use_cuda),
    "bf16": False,
    "dataloader_num_workers": 0,
    "report_to": "none",
}

# TRL versions differ: some use max_length, older versions used max_seq_length.
if "max_length" in available:
    kwargs["max_length"] = MAX_LENGTH
elif "max_seq_length" in available:
    kwargs["max_seq_length"] = MAX_LENGTH

if "eval_strategy" in available:
    kwargs["eval_strategy"] = "epoch"
elif "evaluation_strategy" in available:
    kwargs["evaluation_strategy"] = "epoch"

if "dataset_text_field" in available:
    kwargs["dataset_text_field"] = "text"

if "dataset_num_proc" in available:
    kwargs["dataset_num_proc"] = 1

training_args = SFTConfig(**kwargs)

print("SFTConfig created successfully.")


SFTConfig created successfully.


In [11]:
# Cell 12 — Create SFTTrainer
trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": split["train"],
    "eval_dataset": split["test"],
}

trainer_params = inspect.signature(SFTTrainer.__init__).parameters

# Newer TRL
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
# Older TRL
elif "tokenizer" in trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = SFTTrainer(**trainer_kwargs)

print("Trainer created successfully.")


Truncating train dataset (num_proc=1): 100%|██████████| 306/306 [00:19<00:00, 15.61 examples/s]
Dropping fully masked examples from train dataset (num_proc=1): 100%|██████████| 306/306 [00:13<00:00, 23.39 examples/s]
Truncating eval dataset (num_proc=1): 100%|██████████| 35/35 [00:13<00:00,  2.64 examples/s]
Dropping fully masked examples from eval dataset (num_proc=1): 100%|██████████| 35/35 [00:13<00:00,  2.52 examples/s]


Trainer created successfully.


In [12]:
# Cell 13 — Train
print("Starting fine-tuning...")
result = trainer.train()

print("\nTraining complete.")
print("Training loss:", result.training_loss)


Starting fine-tuning...


c:\Users\tagbi\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
You are not running the flash-attention implementation, expect numerical differences.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# Cell 14 — Save adapter and tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

metrics = {
    "base_model": BASE_MODEL,
    "train_loss": float(result.training_loss),
    "epochs": EPOCHS,
}

with open(os.path.join(OUTPUT_DIR, "training_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("Saved to:", OUTPUT_DIR)
print(metrics)


In [ ]:
# Cell 15 — Quick inference test
model.eval()

question = "What are the early warning signs of liver disease?"

messages = [
    {
        "role": "system",
        "content": (
            "You are LiverAI, a helpful liver-health assistant. "
            "Give safe, concise information and recommend a qualified "
            "health professional when appropriate."
        ),
    },
    {
        "role": "user",
        "content": question,
    },
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

# With device_map='auto', move inputs to the model's first device.
input_device = next(model.parameters()).device
inputs = {k: v.to(input_device) for k, v in inputs.items()}

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.3,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
    )

new_tokens = output[0][inputs["input_ids"].shape[1]:]
answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Question:", question)
print("\nAnswer:", answer)


In [ ]:
# Cell 16 — Save LoRA adapter locally

import os
import shutil

# Folder where the trained LoRA adapter will be saved
OUTPUT_DIR = "./liver-lora"

# Create a ZIP file in the current project directory
zip_base = os.path.abspath("./liver-lora")

shutil.make_archive(
    zip_base,
    "zip",
    OUTPUT_DIR
)

zip_file = zip_base + ".zip"

print("LoRA adapter saved to:")
print(os.path.abspath(OUTPUT_DIR))

print("\nZIP file created at:")
print(zip_file)

## What was wrong in the original notebook

### 1. Model mismatch
The configuration says Phi-3:

`microsoft/Phi-3-mini-4k-instruct`

but Cell 6 uses:

`unsloth/llama-3-8b-Instruct`

and then loads weights from a Windows folder. A Llama checkpoint cannot safely be paired with a Phi-3 configuration/tokenizer.

### 2. Hard-coded Windows path
The original path:

`F:\Liver Disease Detectioon system 2\notebooks`

will fail in Google Colab because that Windows drive does not exist there.

### 3. `config=HF_MODEL_NAME` is the wrong approach here
The model loader should receive the actual model repository or a complete local model directory containing its own `config.json`. The fixed notebook loads the same model from one source.

### 4. TRL API changed
Recent TRL releases changed some SFT argument names. The fixed notebook detects whether the installed version expects `max_length` or `max_seq_length`, and `eval_strategy` or `evaluation_strategy`.

### 5. LoRA target detection was too broad
The original code could select arbitrary linear layer names. The fixed version checks the expected Phi-3 projection modules before applying LoRA.

### 6. The uploaded notebook does not contain saved traceback outputs
The notebook cells contain code, but no actual Colab/console traceback output was stored in the `.ipynb`. Therefore the exact runtime error cannot be recovered from the notebook alone. The fixes above address the concrete code-level problems visible in the notebook.
